In [ ]:
library(Seurat)
library(ggplot2)
library(stringr)
library(dplyr)


# 1. Load nod, step1.sctBeforeIntegrate_splitByMouse_nod.rds

In [ ]:
# scdata=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/3.integrated/scdataDF_merged.rds")
# scdata.nod <- subset(scdata, sampleType=="NODSCID")
# scdata.nod[["RNA"]] = split(scdata.nod[["RNA"]], f = scdata.nod$mouse)

# scdata.nod = SCTransform(scdata.nod,vst.flavor = "v2",vars.to.regress =c("percent.mt"),variable.features.n=3000)
# saveRDS(scdata.nod,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step1.sctBeforeIntegrate_splitByMouse_nod.rds")

# 2. integration by harmony

In [ ]:
scdata.nod=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step1.sctBeforeIntegrate_splitByMouse_nod.rds")
scdata.nod = RunPCA(scdata.nod)
scdata.nod = FindNeighbors(scdata.nod, dims = 1:50, reduction = "pca")
scdata.nod = FindClusters(scdata.nod, resolution = 1, cluster.name = "unintegrated_clusters")
scdata.nod = RunUMAP(scdata.nod, dims = 1:50, reduction = "pca", reduction.name = "umap.unintegrated")

scdata.nod = IntegrateLayers(object = scdata.nod, method = HarmonyIntegration, normalization.method = "SCT", assay = "SCT",
                        orig.reduction = "pca", new.reduction = "harmony",#max.iter.cluster = 200L,npcs = 50L,
                        verbose = F)


scdata.nod = RunUMAP(scdata.nod, reduction = "harmony", dims = 1:50, reduction.name = "umap.harmony",verbose = F)
scdata.nod = FindNeighbors(scdata.nod, reduction = "harmony", dims = 1:50,verbose = F)
scdata.nod = FindClusters(scdata.nod, resolution = 0.1,cluster.name = "harmony_cluster",verbose = F)

saveRDS(scdata.nod,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step2.integrate_splitByMouse_nod.rds")


# 3. remove immune cluster and endothelial

In [ ]:
scdata.nod=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step2.integrate_splitByMouse_nod.rds")
scdata.nod

In [ ]:
options(repr.plot.width=5,repr.plot.height=4.5)
DimPlot(scdata.nod,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=TRUE,label = TRUE,cols=c('0'="#00A087FF",'1'="#4DBBD5FF",'2'="#91D1C2FF",'3'="#3C5488FF",'4'="#F39B7FFF",'5'="#8491B4FF",
  '6'="#fb8072",'7'="#DC0000FF",'8'="#7E6148FF",'9'="#B09C85FF",'10'="#AB82FF",'11'="#00CD00",'12'="#FF6A6A"))

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10) 
VlnPlot(scdata.nod,features=c('Epcam','Ptprc','Pecam1','SMALT-Target','Col1a1','Col1a2'),ncol=3)

In [ ]:
scdata.nod@meta.data[1:2,]

In [ ]:
table(scdata.nod$harmony_cluster)
immune <- subset(scdata.nod,subset = harmony_cluster ==7)
immune@meta.data["harmony_cluster"]=NULL
immune = DietSeurat(immune)
saveRDS(immune,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step3.integrate_splitByMouse_nod_immune.rds")
immune

In [ ]:
scdata <- subset(scdata.nod,subset = harmony_cluster !=7)
table(scdata$harmony_cluster)
scdata@meta.data[1:2,]

In [ ]:
scdata@meta.data["unintegrated_clusters"]=NULL
scdata@meta.data["harmony_cluster"]=NULL
scdata = DietSeurat(scdata)
saveRDS(scdata,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step3.integrate_splitByMouse_nod_rmImmuneEndothelial.rds")

In [ ]:
scdata=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step3.integrate_splitByMouse_nod_rmImmuneEndothelial.rds")
# scdata = DietSeurat(scdata)
scdata[["RNA"]] <- JoinLayers(scdata[["RNA"]])
scdata[["RNA"]] = split(scdata[["RNA"]], f = scdata$mouse)
scdata = SCTransform(scdata,vst.flavor = "v2",vars.to.regress =c("percent.mt", "S.Score", "G2M.Score"),variable.features.n=3000)
saveRDS(scdata,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step4.integrate_splitByMouse_nod_rmImmuneEndothelial_splitByMouse.rds")

# 4. FindNeighbors: dims=1:20; FindClusters: resolution = 0.2; RunUMAP:dims = 1:20

In [ ]:
scdata = FindNeighbors(scdata, reduction = "harmony", dims = 1:20,verbose = T)
scdata = FindClusters(scdata, resolution = 0.2,cluster.name = "harmony_cluster",verbose = F)
scdata = RunUMAP(scdata, reduction = "harmony", dims = 1:20, reduction.name = "umap.harmony",verbose = F)
mycols=c('0'="#00A087FF",'1'="#4DBBD5FF",'2'="#91D1C2FF",'3'="#FF6A6A",'4'="#F39B7FFF",'5'="#8491B4FF",
  '6'="#FF8C00",'7'="#DC0000FF",'8'="#7E6148FF",'9'="#B09C85FF",'10'="#AB82FF",'11'="#00CD00")

options(repr.plot.width=5,repr.plot.height=4.5)
DimPlot(scdata,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=FALSE,label = TRUE,cols=mycols)

options(repr.plot.width =8, repr.plot.height =4)
scdata@active.ident <- factor(Idents(scdata), levels = rev(levels(Idents(scdata))))
p <- DotPlot(scdata, features = c("Tspan1","S100a14","Krt18","Cldn7","Cdh1","Zeb1","Dpp10","Efna5","Tmem163","Syt1",
                                  "H3c3","H2ac4","H3c2","H2ac10","H1f5","Cdc20","Cenpa","Ccnb1","Cdkn3","Ube2c")) +
     RotatedAxis() +
     theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
 labs(title = "", x = "", y = "")
p


x <- scdata@meta.data[,c(1,ncol(scdata@meta.data))]
x$cellID <- row.names(x)
print(length(unique(x$cellID)))
x$cellBC <- sapply(str_split(row.names(x),"_"),"[",3)
x$cellBC <- sapply(str_split(x$cellBC,"-"),"[",1)
rename=read.delim("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/rename.txt",header = T)
x <- merge(x, rename,by="sampleID",all=F)
x <- x[,c(3,4,7,6,1,5,2)]
x$cellID <- gsub("-1$","",x$cellID)
print(length(unique(x$cellID)))
x[1:3,]
write.table(x, file="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step5.scdata_metadata_nod_1_20_0.2.txt",sep="\t", quote=FALSE,row.names=F)


In [ ]:
mycols=c('0'="#FF6A6A",'1'="#00A087FF",'2'="#4DBBD5FF",'3'="#F39B7FFF",
         '4'="#FF8C00",'5'="#00CD00",'6'="#AB82FF",'7'="#DC0000FF",'8'="#7E6148FF")
options(repr.plot.width=5,repr.plot.height=4.5)
DimPlot(scdata,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=FALSE,label = TRUE,cols=mycols)

options(repr.plot.width =6, repr.plot.height =4)
scdata@active.ident <- factor(Idents(scdata), levels = rev(levels(Idents(scdata))))
p <- DotPlot(scdata, features =c("Cdh1","Cldn3","Cldn7","Krt8","Krt18","Cdh2","Dpp6","Dpp10","Dcc","Efna5","Zeb1","Tubb3")) +
     RotatedAxis() +
     theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
 labs(title = "", x = "", y = "")
p


In [ ]:
scdata=readRDS("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step7.scdata_node_w_scores.rds")
scdata

In [ ]:
mycols=c('0'="#FF6A6A",'1'="#00A087FF",'2'="#4DBBD5FF",'3'="#F39B7FFF",
         '4'="#FF8C00",'5'="#00CD00",'6'="#AB82FF",'7'="#DC0000FF",'8'="#7E6148FF")
options(repr.plot.width=5,repr.plot.height=4.5)
DimPlot(scdata,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=FALSE,label = TRUE,cols=mycols)


In [ ]:
scdata = RunUMAP(scdata, reduction = "harmony", dims = 1:50, reduction.name = "umap.harmony",verbose = F)
scdata
mycols=c('0'="#FF6A6A",'1'="#00A087FF",'2'="#4DBBD5FF",'3'="#F39B7FFF",
         '4'="#FF8C00",'5'="#00CD00",'6'="#AB82FF",'7'="#DC0000FF",'8'="#7E6148FF")
options(repr.plot.width=5,repr.plot.height=4.5)
DimPlot(scdata,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=FALSE,label = TRUE,cols=mycols)


In [ ]:
DimPlot(scdata,reduction = "umap.harmony",ncol=1,
  group.by = c("harmony_cluster"),raster=FALSE,label = TRUE,cols=mycols)


# 5. FindAllMarkers

In [ ]:
scdata <- SetIdent(scdata, value = scdata@meta.data$harmony_cluster)
scdata=PrepSCTFindMarkers(scdata)
markers=FindAllMarkers(scdata,test.use = "wilcox",only.pos = T,logfc.threshold = 0.25, assay = "SCT", verbose = F)
write.table(markers,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/nod_markers.csv",sep = "\t",col.names = T,quote=F)

In [ ]:
library(dplyr)
library(ggplot2)
markers <- read.delim("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/nod_markers.csv",header = T)
markers %>%
    group_by(cluster) %>%
    dplyr::filter(avg_log2FC > 1) %>%
    slice_head(n = 5) %>%
    ungroup() -> top5
scdata@active.ident <- factor(Idents(scdata), levels = rev(levels(Idents(scdata))))
p2 <- DotPlot(scdata, features = top5$gene) +
     RotatedAxis() +
     theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
 labs(title = "", x = "", y = "")
p2

In [ ]:
markers %>%
    group_by(cluster) %>%
    dplyr::filter(avg_log2FC > 1) %>%
    arrange(desc(avg_log2FC)) %>%
    slice_head(n = 10) %>%
    ungroup() -> top10
write.table(top10, file="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/nod_markers_top10.csv",sep = ",",col.names = T,quote=F)

# 6. add clone ID

In [ ]:
library(data.table)
mice = c("C4007","C4011","C4033")
INDIR="/syn2/zhaolian/3.JiLab/results/1.BarcodeSeq/04.clone/"
dfiltered <- data.table()
for(my.mouse in mice){
    d <- read.delim(paste0(INDIR, my.mouse, "_umi_table_filtered.txt"))
    d <- data.table(mouse=my.mouse, cellBC=unique(d$cellBC))
    dfiltered <- rbind(dfiltered, d)
}
dclones <- data.table()
for(my.mouse in mice){
    d <- read.delim(paste0(INDIR, my.mouse, "_assigned_clones_clean.txt"))
    d <- data.table(mouse=my.mouse, cellBC=unique(d$cellBC))
    dclones <- rbind(dclones, d)
}
dfiltered[,cellBC:=paste0(mouse,"_",cellBC,"-1")]
dclones$sample <- sapply(strsplit(dclones$cellBC,"_"),"[",1)
dclones$clone <- sapply(strsplit(dclones$cellBC,"_"),"[",3)
dclones$cellBC <- sapply(strsplit(dclones$cellBC,"_"),"[",2)
dclones[,cellBC:=paste0(mouse,"_",sample,"_",cellBC,"-1")]

scdata@meta.data["target"] <- ifelse(row.names(scdata@meta.data) %in% dfiltered$cellBC,"Y","N")
scdata@meta.data["inclone"] <- ifelse(row.names(scdata@meta.data) %in% dclones$cellBC,"Y","N")
scdata@meta.data["clone"]="Others"
for(i in unique(dclones$clone)){scdata@meta.data$clone[which(row.names(scdata@meta.data) %in% dclones[clone==i,cellBC])] <- i}
scdata@meta.data["umapharmony_1"] <- as.data.frame(scdata@reductions$umap.harmony@cell.embeddings)$umapharmony_1
scdata@meta.data["umapharmony_2"] <- as.data.frame(scdata@reductions$umap.harmony@cell.embeddings)$umapharmony_2


In [ ]:
scdata@meta.data[1:2,]

# 7. add mutation info

In [ ]:
library(stringr)
mice=c("Q25","Q186","Q192","C4026","C4007","C4011","C4033")
dt <- data.table()
for(my.mouse in mice[5:7]){
    INDIR=file.path("/syn2/zhaolian/3.JiLab/results/3.PacBio/4.clones_filtered", my.mouse)
    clonelist=read.delim(paste0("/syn2/zhaolian/3.JiLab/results/3.PacBio/4.clones_filtered/",my.mouse, "_clone_list_filtered.txt"),header = T)
    for(i in clonelist$clone){
        phy=read.table(paste0(INDIR,"/",i,"_filtered.phy"),sep=" ",header=F, skip=2,colClasses = c("character","character"),col.names=c("cellBC","bi"))
        setDT(phy)
        phy[,numMut := str_count(bi,"1")]
        dt <- rbind(dt, phy)
    }
}
dt[1:2,]
nrow(dt)
length(unique(dt$cellBC))
#####################################
dt[,cellBC := paste0(cellBC,"-1")]
setDF(dt)
row.names(dt) <- dt$cellBC
scdata@meta.data["numMut"] <- "Others"
scdata@meta.data[row.names(scdata@meta.data) %in% rownames(dt),'numMut'] <- dt[row.names(scdata@meta.data)[row.names(scdata@meta.data) %in% rownames(dt)],'numMut']
#####################################
hist(dt$numMut, breaks=100)


In [ ]:
saveRDS(scdata,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step6.scdata_node_clone_numMut.rds")

# 8. add MetGroup to scdata and export step7.*.rds

In [ ]:
scdata@meta.data[1:2,]
row_annotation_data <- read.table("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byType/step8.nod_row_annotation_data0.98.txt")
clone_matrix<- read.table("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byType/step8.clone_matrix0.98.txt")
row_annotation_data[1:2,]

In [ ]:
scdata@meta.data[1:2,]
row_annotation_data <- read.table("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byType/step8.nod_row_annotation_data0.98.txt")
clone_matrix<- read.table("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byType/step8.clone_matrix0.98.txt")
row_annotation_data[1:2,]

In [ ]:
###################################################
## 1. add mouse_clone
###################################################
scdata@meta.data$mouse_clone <- "Others"
non_others_idx <- scdata@meta.data$clone != "Others"
scdata@meta.data$mouse_clone[non_others_idx] <- paste0(scdata@meta.data$mouse[non_others_idx], "_", scdata@meta.data$clone[non_others_idx])
###################################################
## 2. add MetStatus to scdata@meta.data
###################################################
scdata@meta.data$MetStatus <- "Others"
matched_indices <- match(scdata@meta.data$mouse_clone, rownames(row_annotation_data))
valid_indices <- !is.na(matched_indices)  # Ensure valid matches
scdata@meta.data$MetStatus[valid_indices] <- row_annotation_data$MetStatus[matched_indices[valid_indices]]
scdata@meta.data[1:2,]
###################################################
## 3. add MetGroup to scdata@meta.data
###################################################
scdata@meta.data$MetGroup <- "Others"
matched_indices <- match(scdata@meta.data$mouse_clone, rownames(row_annotation_data))
valid_indices <- !is.na(matched_indices)  # Ensure valid matches
scdata@meta.data$MetGroup[valid_indices] <- row_annotation_data$MetGroup[matched_indices[valid_indices]]

###################################################
## 4. add scFitness to scdata@meta.data
###################################################
mice = c("C4007","C4011","C4033")
INDIR="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/5.2.scFitness/"
dfitness <- data.frame()
for(my.mouse in mice){
    d <- read.table(paste0(INDIR, my.mouse, "_normalized_mean_fitness.txt"), header = FALSE, skip=2)
    d <- d[d$V1 != "ref",]
    d$V1 <- paste0(d$V1,"-1")
    row.names(d) <- d$V1
    dfitness <- rbind(dfitness,d)
}
scdata@meta.data["scFitness"]="Others"
scdata@meta.data[row.names(scdata@meta.data) %in% rownames(dfitness),'scFitness'] <- dfitness[row.names(scdata@meta.data)[row.names(scdata@meta.data) %in% rownames(dfitness)],'V3']
###################################################
## 5. add scPlasticity to scdata@meta.data
###################################################
INDIR="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/5.3.scPlasticity/"
dplasticity <- data.frame()
for(my.mouse in mice){
    d <- read.table(paste0(INDIR, my.mouse, "_plasticity.txt"), header = TRUE)
    d$cellID <- paste0(d$cellID,"-1")
    row.names(d) <- d$cellID
    dplasticity <- rbind(dplasticity,d)
}
scdata@meta.data["scPlasticity"]="Others"
scdata@meta.data[row.names(scdata@meta.data) %in% rownames(dplasticity),'scPlasticity'] <- dplasticity[row.names(scdata@meta.data)[row.names(scdata@meta.data) %in% rownames(dplasticity)],'scPlasticity']
###################################################
## 6. add scMetRate to scdata@meta.data
###################################################
INDIR="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/5.4.scMetRate/"
dmetrate <- data.frame()
for(my.mouse in mice){
    d <- read.table(paste0(INDIR, "metatable_",my.mouse, ".tsv"), header = TRUE)
    d <- d[d$original_Cell != "ref",]
    d$original_Cell <- paste0(d$original_Cell,"-1")
    row.names(d) <- d$original_Cell
    dmetrate <- rbind(dmetrate,d)
}
scdata@meta.data["scMetRate"]="Others"
scdata@meta.data[row.names(scdata@meta.data) %in% rownames(dmetrate),'scMetRate'] <- dmetrate[row.names(scdata@meta.data)[row.names(scdata@meta.data) %in% rownames(dmetrate)],'scMetRate']
###################################################
## 7. add cytoTrace to scdata@meta.data
###################################################
dcytoTrace <- read.delim("/syn2/zhaolian/3.JiLab/results/2.scRNAseq/5.6.cytoTrace/root_to_tip_scPlasticity.tsv")
dcytoTrace$Cell_barcode	<- paste0(dcytoTrace$Cell_barcode,"-1")
dcytoTrace <- dcytoTrace[,c(1:5)]
dcytoTrace <- dcytoTrace[!duplicated(dcytoTrace),]
row.names(dcytoTrace) <- dcytoTrace$Cell_barcode
scdata@meta.data["Root_to_tip_distance"]="Others"
scdata@meta.data[row.names(scdata@meta.data) %in% rownames(dcytoTrace),'Root_to_tip_distance'] <- 
        dcytoTrace[row.names(scdata@meta.data)[row.names(scdata@meta.data) %in% rownames(dcytoTrace)],'Root_to_tip_distance']
scdata@meta.data["CytoTRACE2_Score"]="Others"
scdata@meta.data[row.names(scdata@meta.data) %in% rownames(dcytoTrace),'CytoTRACE2_Score'] <- 
        dcytoTrace[row.names(scdata@meta.data)[row.names(scdata@meta.data) %in% rownames(dcytoTrace)],'CytoTRACE2_Score']
scdata@meta.data["CytoTRACE2_Potency"]="Others"
scdata@meta.data[row.names(scdata@meta.data) %in% rownames(dcytoTrace),'CytoTRACE2_Potency'] <- 
        dcytoTrace[row.names(scdata@meta.data)[row.names(scdata@meta.data) %in% rownames(dcytoTrace)],'CytoTRACE2_Potency']
scdata@meta.data["CytoTRACE2_Relative"]="Others"
scdata@meta.data[row.names(scdata@meta.data) %in% rownames(dcytoTrace),'CytoTRACE2_Relative'] <- 
        dcytoTrace[row.names(scdata@meta.data)[row.names(scdata@meta.data) %in% rownames(dcytoTrace)],'CytoTRACE2_Relative']
###################################################
scdata@meta.data[1:5,]
saveRDS(scdata,"/syn2/zhaolian/3.JiLab/results/2.scRNAseq/4.integrated_byTypeReClustering/step7.scdata_node_w_scores.rds")